# Phase 1: Data Diagnostics & Intelligence (EDA)

**Objective:** Analyze dataset quality before YOLO conversion to prevent "learning noise."

---

## Table of Contents

1. [Configuration & Setup](#config)
2. [Load & Parse COCO JSON](#load)
3. [Dataset Overview Dashboard](#overview)
4. [Image Dimension Analysis](#dimensions)
5. [Area & Aspect Ratio Analysis](#area-ar)
6. [Spatial Distribution Heatmap](#heatmap)
7. [Hierarchical Consistency Check](#hierarchy)
8. [Overlap / IoU Analysis](#iou)
9. [Negative Sample Strategy](#negatives)
10. [Outlier Sample Visualization](#outlier-viz)
11. [Summary Report & Recommendations](#summary)

---

<a id='config'></a>
## 1. Configuration & Setup

In [ ]:
import json
import os
import warnings
from pathlib import Path
from collections import defaultdict
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle
import seaborn as sns
from PIL import Image

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# -------------------- PATHS --------------------
JSON_PATH = '/kaggle/input/datasets/mohammedahmedxx12/machathon-dataset/Phase1_Train_Dataset/annotations/Cells_Anotations_coco.json'
IMAGES_DIR = '/kaggle/input/datasets/mohammedahmedxx12/machathon-dataset/Phase1_Train_Dataset/images'
OUTPUT_DIR = '/kaggle/working'

# -------------------- THRESHOLDS --------------------
TINY_AREA_PCT = 0.03        # Boxes < 3% of image area are flagged (for tables only)
FULL_PAGE_PCT = 0.95        # Boxes > 95% of image area are flagged
EXTREME_AR_LOW = 0.2        # Aspect ratio < 0.2 (extremely tall)
EXTREME_AR_HIGH = 5.0       # Aspect ratio > 5.0 (extremely wide)
NEGATIVE_RATIO_MIN = 0.10   # Target 10% minimum empty images
NEGATIVE_RATIO_MAX = 0.15   # Target 15% maximum empty images
IOU_DUPLICATE_THRESH = 0.8  # IoU > 0.8 between same-category boxes = duplicate

# -------------------- CATEGORY MAPPING --------------------
# Will be populated after loading JSON
CAT_ID_TO_NAME = {}
CAT_NAME_TO_ID = {}

# -------------------- COLOR PALETTE --------------------
CATEGORY_COLORS = {
    1: '#E74C3C',  # table - red
    2: '#3498DB',  # table column - blue
    3: '#2ECC71',  # table row - green
    4: '#9B59B6',  # table column header - purple
    5: '#F39C12',  # table projected row header - orange
    6: '#1ABC9C',  # table spanning cell - teal
}

# Seaborn style
sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("loaded")

<a id='load'></a>
## 2. Load & Parse COCO JSON

In [ ]:
with open(JSON_PATH, 'r') as f:
    coco_data = json.load(f)

# Extract components
images_list = coco_data['images']
annotations_list = coco_data['annotations']
categories_list = coco_data['categories']

print(f"   Loaded {len(images_list):,} images")
print(f"   Loaded {len(annotations_list):,} annotations")
print(f"   Loaded {len(categories_list)} categories")

# -------------------- Build DataFrames --------------------

# Categories DataFrame
df_cats = pd.DataFrame(categories_list)
CAT_ID_TO_NAME = dict(zip(df_cats['id'], df_cats['name']))
CAT_NAME_TO_ID = dict(zip(df_cats['name'], df_cats['id']))

print("\n Categories:")
for _, row in df_cats.iterrows():
    print(f"   {row['id']}: {row['name']} (parent: {row['supercategory']})")

# Images DataFrame
df_images = pd.DataFrame(images_list)
df_images['image_area'] = df_images['width'] * df_images['height']
df_images = df_images.rename(columns={'id': 'image_id'})

# Annotations DataFrame
df_anns = pd.DataFrame(annotations_list)

# Decode bbox [x, y, width, height] into separate columns
df_anns['bbox_x'] = df_anns['bbox'].apply(lambda b: b[0])
df_anns['bbox_y'] = df_anns['bbox'].apply(lambda b: b[1])
df_anns['bbox_w'] = df_anns['bbox'].apply(lambda b: b[2])
df_anns['bbox_h'] = df_anns['bbox'].apply(lambda b: b[3])

# Compute center coordinates
df_anns['bbox_cx'] = df_anns['bbox_x'] + df_anns['bbox_w'] / 2
df_anns['bbox_cy'] = df_anns['bbox_y'] + df_anns['bbox_h'] / 2

# -------------------- Merge with Images --------------------
df_anns = df_anns.merge(df_images[['image_id', 'file_name', 'width', 'height', 'image_area']], 
                        on='image_id', how='left')

# -------------------- Derived Columns --------------------
df_anns['box_area_pct'] = df_anns['area'] / df_anns['image_area']
df_anns['aspect_ratio'] = df_anns['bbox_w'] / df_anns['bbox_h'].replace(0, np.nan)
df_anns['category_name'] = df_anns['category_id'].map(CAT_ID_TO_NAME)

# Normalized center (0-1 range)
df_anns['norm_cx'] = df_anns['bbox_cx'] / df_anns['width']
df_anns['norm_cy'] = df_anns['bbox_cy'] / df_anns['height']

print("\n DataFrames built successfully.")
print(f"   df_images shape: {df_images.shape}")
print(f"   df_anns shape: {df_anns.shape}")
print(f"   df_cats shape: {df_cats.shape}")

# Quick sanity check
print("\n Sample annotation record:")
display(df_anns.head(1).T)

<a id='overview'></a>
## 3. Dataset Overview Dashboard

In [ ]:
# ============================================================
# CELL 2: Dataset Overview Dashboard
# ============================================================

print("="*60)
print(" DATASET OVERVIEW")
print("="*60)

n_images = len(df_images)
n_annotations = len(df_anns)
n_categories = len(df_cats)

print(f"\n Total Images: {n_images:,}")
print(f" Total Annotations: {n_annotations:,}")
print(f" Avg Annotations/Image: {n_annotations/n_images:.1f}")
print(f" Categories: {n_categories}")

# -------------------- Annotation Count per Category --------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: Annotations per category
cat_counts = df_anns['category_name'].value_counts().sort_index()
colors = [CATEGORY_COLORS.get(CAT_NAME_TO_ID.get(name, 1), '#333333') for name in cat_counts.index]

ax1 = axes[0]
bars = ax1.bar(cat_counts.index, cat_counts.values, color=colors, edgecolor='black', linewidth=0.5)
ax1.set_xlabel('Category', fontsize=12)
ax1.set_ylabel('Annotation Count', fontsize=12)
ax1.set_title('Annotation Count per Category (Class Imbalance Check)', fontsize=13, fontweight='bold')
ax1.tick_params(axis='x', rotation=45)

# Add count labels on bars
for bar, count in zip(bars, cat_counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100, 
             f'{count:,}', ha='center', va='bottom', fontsize=10)

# -------------------- Annotations per Image Distribution --------------------
anns_per_image = df_anns.groupby('image_id').size()

ax2 = axes[1]
ax2.hist(anns_per_image, bins=50, color='#3498DB', edgecolor='black', alpha=0.7)
ax2.axvline(anns_per_image.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {anns_per_image.mean():.1f}')
ax2.axvline(anns_per_image.median(), color='orange', linestyle='--', linewidth=2, label=f'Median: {anns_per_image.median():.1f}')
ax2.set_xlabel('Annotations per Image', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.set_title('Distribution of Annotations per Image', fontsize=13, fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.show()

# -------------------- Flag Outlier Images --------------------
mean_anns = anns_per_image.mean()
std_anns = anns_per_image.std()
threshold_high = mean_anns + 3 * std_anns
threshold_low = 3  # Images with very few annotations

high_ann_images = anns_per_image[anns_per_image > threshold_high]
low_ann_images = anns_per_image[anns_per_image < threshold_low]

print(f"\n  Outlier Detection (Annotations per Image):")
print(f"   Mean: {mean_anns:.1f}, Std: {std_anns:.1f}")
print(f"   High threshold (>3σ): {threshold_high:.1f}")
print(f"   Images with >3σ annotations: {len(high_ann_images)}")
print(f"   Images with <3 annotations: {len(low_ann_images)}")

if len(high_ann_images) > 0:
    print(f"\n    High annotation count images (top 5):")
    for img_id, count in high_ann_images.nlargest(5).items():
        fname = df_images[df_images['image_id'] == img_id]['file_name'].values[0]
        print(f"      {fname}: {count} annotations")

<a id='dimensions'></a>
## 4. Image Dimension Analysis

In [ ]:
# ============================================================
# CELL 3: Image Dimension Analysis
# ============================================================

print("="*60)
print(" IMAGE DIMENSION ANALYSIS")
print("="*60)

# Check dimension variance
unique_dims = df_images.groupby(['width', 'height']).size().reset_index(name='count')
unique_dims = unique_dims.sort_values('count', ascending=False)

print(f"\n Unique Image Dimensions: {len(unique_dims)}")
print("\n   Top 10 most common dimensions:")
for _, row in unique_dims.head(10).iterrows():
    pct = row['count'] / n_images * 100
    print(f"      {row['width']}x{row['height']}: {row['count']:,} images ({pct:.1f}%)")

# -------------------- Visualization --------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot of dimensions
ax1 = axes[0]
scatter = ax1.scatter(df_images['width'], df_images['height'], 
                      alpha=0.5, c='#3498DB', edgecolors='black', linewidth=0.3, s=50)
ax1.set_xlabel('Width (pixels)', fontsize=12)
ax1.set_ylabel('Height (pixels)', fontsize=12)
ax1.set_title('Image Dimensions Scatter Plot', fontsize=13, fontweight='bold')

# Add most common dimension marker
most_common = unique_dims.iloc[0]
ax1.scatter(most_common['width'], most_common['height'], 
            c='red', s=200, marker='*', zorder=5, label=f"Most common: {most_common['width']}x{most_common['height']}")
ax1.legend()

# Aspect ratio distribution of images
ax2 = axes[1]
img_aspect_ratios = df_images['width'] / df_images['height']
ax2.hist(img_aspect_ratios, bins=30, color='#2ECC71', edgecolor='black', alpha=0.7)
ax2.axvline(img_aspect_ratios.mean(), color='red', linestyle='--', linewidth=2, 
            label=f'Mean AR: {img_aspect_ratios.mean():.2f}')
ax2.set_xlabel('Image Aspect Ratio (W/H)', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.set_title('Image Aspect Ratio Distribution', fontsize=13, fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.show()

# -------------------- Input Resolution Recommendation --------------------
mean_width = df_images['width'].mean()
mean_height = df_images['height'].mean()
mean_ar = mean_width / mean_height

print(f"\n Image Statistics:")
print(f"   Mean Width: {mean_width:.0f}px")
print(f"   Mean Height: {mean_height:.0f}px")
print(f"   Mean Aspect Ratio: {mean_ar:.3f}")

# Recommendation based on aspect ratio
if 0.9 <= mean_ar <= 1.1:
    rec_res = "640x640 or 1280x1280 (square)"
elif mean_ar < 0.9:
    # Taller images
    rec_res = "640x960 or 640x1280 (portrait)"
else:
    # Wider images
    rec_res = "960x640 or 1280x640 (landscape)"

print(f"\n Recommended YOLOv10 Input Resolution: {rec_res}")

<a id='area-ar'></a>
## 5. Area & Aspect Ratio Analysis (Core Requirement 1)

In [ ]:
# ============================================================
# CELL 4a: Box Area vs. Image Area Analysis
# ============================================================

print("="*60)
print(" AREA ANALYSIS: Box Area % of Image")
print("="*60)

# -------------------- Statistics by Category --------------------
print("\n Box Area Percentage Statistics by Category:")
area_stats = df_anns.groupby('category_name')['box_area_pct'].describe()
display(area_stats.round(4))

# -------------------- Scatter Plot: Box Area % vs Image Area --------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax1 = axes[0]
for cat_id, cat_name in CAT_ID_TO_NAME.items():
    subset = df_anns[df_anns['category_id'] == cat_id]
    ax1.scatter(subset['image_area'], subset['box_area_pct'] * 100, 
                alpha=0.4, label=cat_name, c=CATEGORY_COLORS.get(cat_id, '#333'),
                s=20, edgecolors='none')

# Threshold lines
ax1.axhline(y=TINY_AREA_PCT * 100, color='orange', linestyle='--', linewidth=2, label=f'Tiny threshold ({TINY_AREA_PCT*100}%)')
ax1.axhline(y=FULL_PAGE_PCT * 100, color='red', linestyle='--', linewidth=2, label=f'Full-page threshold ({FULL_PAGE_PCT*100}%)')

ax1.set_xlabel('Image Area (pixels²)', fontsize=12)
ax1.set_ylabel('Box Area (% of Image)', fontsize=12)
ax1.set_title('Box Area % vs Image Area (by Category)', fontsize=13, fontweight='bold')
ax1.legend(loc='upper right', fontsize=8)
ax1.set_ylim(0, 105)

# -------------------- KDE Distribution by Category --------------------
ax2 = axes[1]
for cat_id, cat_name in CAT_ID_TO_NAME.items():
    subset = df_anns[df_anns['category_id'] == cat_id]
    if len(subset) > 10:
        sns.kdeplot(data=subset['box_area_pct'] * 100, ax=ax2, 
                    label=cat_name, color=CATEGORY_COLORS.get(cat_id, '#333'),
                    linewidth=2)

ax2.axvline(x=TINY_AREA_PCT * 100, color='orange', linestyle='--', linewidth=2)
ax2.axvline(x=FULL_PAGE_PCT * 100, color='red', linestyle='--', linewidth=2)
ax2.set_xlabel('Box Area (% of Image)', fontsize=12)
ax2.set_ylabel('Density', fontsize=12)
ax2.set_title('Box Area % Distribution by Category (KDE)', fontsize=13, fontweight='bold')
ax2.legend(loc='upper right', fontsize=9)
ax2.set_xlim(0, 100)

plt.tight_layout()
plt.show()

# -------------------- Flag Tiny Boxes (TABLES ONLY) --------------------
table_cat_id = CAT_NAME_TO_ID.get('table', 1)
df_tiny_tables = df_anns[(df_anns['category_id'] == table_cat_id) & 
                          (df_anns['box_area_pct'] < TINY_AREA_PCT)].copy()

print(f"\n  TINY TABLE BOXES (<{TINY_AREA_PCT*100}% of page):")
print(f"   Found: {len(df_tiny_tables)} annotations")
if len(df_tiny_tables) > 0:
    print("   Sample flagged files:")
    for _, row in df_tiny_tables.head(5).iterrows():
        print(f"      - {row['file_name']} (ann_id: {row['id']}, area%: {row['box_area_pct']*100:.2f}%)")

# -------------------- Flag Full-Page Boxes (ALL CATEGORIES) --------------------
df_fullpage = df_anns[df_anns['box_area_pct'] > FULL_PAGE_PCT].copy()

print(f"\n  FULL-PAGE BOXES (>{FULL_PAGE_PCT*100}% of page):")
print(f"   Found: {len(df_fullpage)} annotations")
if len(df_fullpage) > 0:
    print("   Sample flagged files:")
    for _, row in df_fullpage.head(5).iterrows():
        print(f"      - {row['file_name']} (ann_id: {row['id']}, cat: {row['category_name']}, area%: {row['box_area_pct']*100:.2f}%)")

In [ ]:
# ============================================================
# CELL 4b: Width-to-Height Ratio (Aspect Ratio) Analysis
# ============================================================

print("="*60)
print(" ASPECT RATIO ANALYSIS: Width / Height")
print("="*60)

# -------------------- Statistics by Category --------------------
print("\n Aspect Ratio Statistics by Category:")
ar_stats = df_anns.groupby('category_name')['aspect_ratio'].describe()
display(ar_stats.round(3))

# -------------------- Histogram + Box Plot --------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Histogram by category
ax1 = axes[0]
for cat_id, cat_name in CAT_ID_TO_NAME.items():
    subset = df_anns[df_anns['category_id'] == cat_id]
    # Clip extreme values for visualization
    ar_clipped = subset['aspect_ratio'].clip(0, 10)
    ax1.hist(ar_clipped, bins=50, alpha=0.5, label=cat_name, 
             color=CATEGORY_COLORS.get(cat_id, '#333'))

ax1.axvline(x=EXTREME_AR_LOW, color='red', linestyle='--', linewidth=2, label=f'Extreme low ({EXTREME_AR_LOW})')
ax1.axvline(x=EXTREME_AR_HIGH, color='red', linestyle='--', linewidth=2, label=f'Extreme high ({EXTREME_AR_HIGH})')
ax1.set_xlabel('Aspect Ratio (W/H)', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title('Aspect Ratio Distribution by Category', fontsize=13, fontweight='bold')
ax1.legend(loc='upper right', fontsize=8)
ax1.set_xlim(0, 10)

# Box plot by category
ax2 = axes[1]
df_anns_ar_plot = df_anns[df_anns['aspect_ratio'].notna()].copy()
df_anns_ar_plot['aspect_ratio_clipped'] = df_anns_ar_plot['aspect_ratio'].clip(0, 10)

order = list(CAT_ID_TO_NAME.values())
palette = [CATEGORY_COLORS.get(CAT_NAME_TO_ID.get(name, 1), '#333') for name in order]

sns.boxplot(data=df_anns_ar_plot, x='category_name', y='aspect_ratio_clipped', 
            order=order, palette=palette, ax=ax2)
ax2.axhline(y=EXTREME_AR_LOW, color='red', linestyle='--', linewidth=2)
ax2.axhline(y=EXTREME_AR_HIGH, color='red', linestyle='--', linewidth=2)
ax2.set_xlabel('Category', fontsize=12)
ax2.set_ylabel('Aspect Ratio (W/H)', fontsize=12)
ax2.set_title('Aspect Ratio Box Plot by Category', fontsize=13, fontweight='bold')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# -------------------- Flag Extreme Aspect Ratio (TABLES ONLY) --------------------
df_extreme_ar = df_anns[(df_anns['category_id'] == table_cat_id) & 
                        ((df_anns['aspect_ratio'] < EXTREME_AR_LOW) | 
                         (df_anns['aspect_ratio'] > EXTREME_AR_HIGH))].copy()

print(f"\n  EXTREME ASPECT RATIO TABLES (AR<{EXTREME_AR_LOW} or AR>{EXTREME_AR_HIGH}):")
print(f"   Found: {len(df_extreme_ar)} annotations")
if len(df_extreme_ar) > 0:
    print("   Sample flagged files:")
    for _, row in df_extreme_ar.head(5).iterrows():
        print(f"      - {row['file_name']} (ann_id: {row['id']}, AR: {row['aspect_ratio']:.3f})")

# -------------------- Input Resolution Recommendation --------------------
table_ar = df_anns[df_anns['category_id'] == table_cat_id]['aspect_ratio']
table_ar_median = table_ar.median()
table_ar_p25 = table_ar.quantile(0.25)
table_ar_p75 = table_ar.quantile(0.75)

print(f"\n TABLE Aspect Ratio Insights:")
print(f"   Median AR: {table_ar_median:.2f}")
print(f"   IQR: [{table_ar_p25:.2f}, {table_ar_p75:.2f}]")

if table_ar_median > 2.0:
    print(f"     Tables are predominantly WIDE. Consider wider input resolution (e.g., 1280x640).")
elif table_ar_median < 0.5:
    print(f"     Tables are predominantly TALL. Consider taller input resolution (e.g., 640x1280).")
else:
    print(f"     Tables have balanced AR. Standard square input (640x640 or 1280x1280) should work well.")

<a id='heatmap'></a>
## 6. Spatial Distribution Heatmap

In [ ]:
# ============================================================
# CELL 5: Spatial Distribution Heatmap
# ============================================================

print("="*60)
print(" SPATIAL DISTRIBUTION HEATMAP")
print("="*60)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# -------------------- All Annotations Heatmap --------------------
ax1 = axes[0]
h1 = ax1.hist2d(df_anns['norm_cx'], df_anns['norm_cy'], bins=30, cmap='YlOrRd', cmin=1)
ax1.set_xlabel('Normalized X (0=left, 1=right)', fontsize=11)
ax1.set_ylabel('Normalized Y (0=top, 1=bottom)', fontsize=11)
ax1.set_title('All Annotations Center Distribution', fontsize=12, fontweight='bold')
ax1.invert_yaxis()  # Top of image at top of plot
ax1.set_xlim(0, 1)
ax1.set_ylim(1, 0)
plt.colorbar(h1[3], ax=ax1, label='Count')

# -------------------- Tables Only Heatmap --------------------
ax2 = axes[1]
df_tables = df_anns[df_anns['category_id'] == table_cat_id]
h2 = ax2.hist2d(df_tables['norm_cx'], df_tables['norm_cy'], bins=20, cmap='Blues', cmin=1)
ax2.set_xlabel('Normalized X', fontsize=11)
ax2.set_ylabel('Normalized Y', fontsize=11)
ax2.set_title('TABLE Annotations Only', fontsize=12, fontweight='bold')
ax2.invert_yaxis()
ax2.set_xlim(0, 1)
ax2.set_ylim(1, 0)
plt.colorbar(h2[3], ax=ax2, label='Count')

# -------------------- Sub-components (non-table) Heatmap --------------------
ax3 = axes[2]
df_subcomponents = df_anns[df_anns['category_id'] != table_cat_id]
h3 = ax3.hist2d(df_subcomponents['norm_cx'], df_subcomponents['norm_cy'], bins=30, cmap='Greens', cmin=1)
ax3.set_xlabel('Normalized X', fontsize=11)
ax3.set_ylabel('Normalized Y', fontsize=11)
ax3.set_title('Sub-components (cells/rows/cols)', fontsize=12, fontweight='bold')
ax3.invert_yaxis()
ax3.set_xlim(0, 1)
ax3.set_ylim(1, 0)
plt.colorbar(h3[3], ax=ax3, label='Count')

plt.tight_layout()
plt.show()

# -------------------- Edge/Margin Analysis --------------------
margin_thresh = 0.05  # 5% from edge

# Check if annotations cluster near edges (possible margin annotations)
near_left = df_anns[df_anns['norm_cx'] < margin_thresh]
near_right = df_anns[df_anns['norm_cx'] > (1 - margin_thresh)]
near_top = df_anns[df_anns['norm_cy'] < margin_thresh]
near_bottom = df_anns[df_anns['norm_cy'] > (1 - margin_thresh)]

print(f"\n Annotations near edges (<{margin_thresh*100}% from edge):")
print(f"   Near left edge: {len(near_left)} ({len(near_left)/len(df_anns)*100:.1f}%)")
print(f"   Near right edge: {len(near_right)} ({len(near_right)/len(df_anns)*100:.1f}%)")
print(f"   Near top edge: {len(near_top)} ({len(near_top)/len(df_anns)*100:.1f}%)")
print(f"   Near bottom edge: {len(near_bottom)} ({len(near_bottom)/len(df_anns)*100:.1f}%)")

print("\n Interpretation: Most annotations should cluster toward the center. ")
print("   High edge counts might indicate margin/border annotations.")

<a id='hierarchy'></a>
## 7. Hierarchical Consistency Check

In [ ]:
# ============================================================
# CELL 6: Hierarchical Consistency Check
# ============================================================

print("="*60)
print(" HIERARCHICAL CONSISTENCY CHECK")
print("="*60)

# Get image IDs with different category types
images_with_tables = set(df_anns[df_anns['category_id'] == table_cat_id]['image_id'].unique())
images_with_children = set(df_anns[df_anns['category_id'] != table_cat_id]['image_id'].unique())

# -------------------- Check 1: Orphaned Children --------------------
# Images that have child annotations (cells, rows, cols) but no table parent
orphaned_children_images = images_with_children - images_with_tables

print(f"\n Check 1: Orphaned Child Annotations")
print(f"   Images with children but no table parent: {len(orphaned_children_images)}")

if len(orphaned_children_images) > 0:
    print("     This is a data quality issue - child annotations exist without parent table!")
    orphan_image_ids = list(orphaned_children_images)[:5]
    print(f"   Sample image IDs: {orphan_image_ids}")
    
    df_orphan_anns = df_anns[df_anns['image_id'].isin(orphaned_children_images)].copy()
else:
    print("    All child annotations have a parent table.")
    df_orphan_anns = pd.DataFrame()

# -------------------- Check 2: Tables without Children --------------------
# Images that have a table but zero child annotations (suspicious incomplete labeling)
tables_without_children = images_with_tables - images_with_children

print(f"\n Check 2: Tables without Child Annotations")
print(f"   Images with table but no children: {len(tables_without_children)}")

if len(tables_without_children) > 0:
    print("     These tables might be incompletely annotated!")
    sample_ids = list(tables_without_children)[:5]
    for img_id in sample_ids:
        fname = df_images[df_images['image_id'] == img_id]['file_name'].values[0]
        print(f"      - {fname}")
else:
    print("    All tables have child annotations.")

# -------------------- Check 3: Spatial Containment --------------------
# Verify child annotations are spatially within the parent table bbox

def check_containment(table_bbox, child_bbox, tolerance=10):
    """Check if child bbox is contained within table bbox (with tolerance in pixels)"""
    tx, ty, tw, th = table_bbox
    cx, cy, cw, ch = child_bbox
    
    # Child must be within table bounds (with some tolerance for annotation noise)
    return (cx >= tx - tolerance and 
            cy >= ty - tolerance and
            cx + cw <= tx + tw + tolerance and
            cy + ch <= ty + th + tolerance)

print(f"\n Check 3: Spatial Containment of Children within Tables")

containment_issues = []

# Group annotations by image
for image_id in images_with_tables:
    img_anns = df_anns[df_anns['image_id'] == image_id]
    
    # Get all table bboxes for this image
    table_bboxes = img_anns[img_anns['category_id'] == table_cat_id]['bbox'].tolist()
    
    # Get all child annotations
    children = img_anns[img_anns['category_id'] != table_cat_id]
    
    for _, child in children.iterrows():
        child_bbox = child['bbox']
        
        # Check if child is contained in ANY table bbox
        is_contained = any(check_containment(tb, child_bbox) for tb in table_bboxes)
        
        if not is_contained:
            containment_issues.append({
                'image_id': image_id,
                'file_name': child['file_name'],
                'ann_id': child['id'],
                'category': child['category_name'],
                'child_bbox': child_bbox
            })

df_containment_issues = pd.DataFrame(containment_issues)

print(f"   Child annotations outside parent table: {len(df_containment_issues)}")
if len(df_containment_issues) > 0:
    print("     These annotations are spatially outside their expected parent!")
    print("   Sample issues:")
    for _, row in df_containment_issues.head(5).iterrows():
        print(f"      - {row['file_name']} (ann_id: {row['ann_id']}, cat: {row['category']})")
else:
    print("    All child annotations are spatially within their parent tables.")

<a id='iou'></a>
## 8. Overlap / IoU Analysis

In [ ]:
# ============================================================
# CELL 7: Overlap / IoU Analysis (Duplicate Detection)
# ============================================================

print("="*60)
print(" IoU ANALYSIS: Duplicate Annotation Detection")
print("="*60)

def compute_iou(bbox1, bbox2):
    """Compute IoU between two bboxes in [x, y, w, h] format"""
    x1, y1, w1, h1 = bbox1
    x2, y2, w2, h2 = bbox2
    
    # Convert to [x1, y1, x2, y2] format
    box1 = [x1, y1, x1 + w1, y1 + h1]
    box2 = [x2, y2, x2 + w2, y2 + h2]
    
    # Compute intersection
    inter_x1 = max(box1[0], box2[0])
    inter_y1 = max(box1[1], box2[1])
    inter_x2 = min(box1[2], box2[2])
    inter_y2 = min(box1[3], box2[3])
    
    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h
    
    # Compute union
    area1 = w1 * h1
    area2 = w2 * h2
    union_area = area1 + area2 - inter_area
    
    if union_area == 0:
        return 0.0
    
    return inter_area / union_area

# Detect potential duplicates (same category, same image, high IoU)
print(f"\n Checking for duplicate annotations (IoU > {IOU_DUPLICATE_THRESH})...")

duplicates = []

# Group by image and category
for (image_id, cat_id), group in df_anns.groupby(['image_id', 'category_id']):
    if len(group) < 2:
        continue
    
    # Check all pairs within this group
    bboxes = group['bbox'].tolist()
    ann_ids = group['id'].tolist()
    
    for i, j in combinations(range(len(bboxes)), 2):
        iou = compute_iou(bboxes[i], bboxes[j])
        if iou > IOU_DUPLICATE_THRESH:
            fname = group['file_name'].iloc[0]
            cat_name = CAT_ID_TO_NAME.get(cat_id, 'unknown')
            duplicates.append({
                'image_id': image_id,
                'file_name': fname,
                'category': cat_name,
                'ann_id_1': ann_ids[i],
                'ann_id_2': ann_ids[j],
                'iou': iou
            })

df_duplicates = pd.DataFrame(duplicates)

print(f"\n  POTENTIAL DUPLICATE ANNOTATIONS (IoU > {IOU_DUPLICATE_THRESH}):")
print(f"   Found: {len(df_duplicates)} pairs")

if len(df_duplicates) > 0:
    print("\n   Breakdown by category:")
    dup_by_cat = df_duplicates['category'].value_counts()
    for cat, count in dup_by_cat.items():
        print(f"      {cat}: {count} pairs")
    
    print("\n   Sample duplicates:")
    for _, row in df_duplicates.head(5).iterrows():
        print(f"      - {row['file_name']} | {row['category']} | IDs: {row['ann_id_1']}, {row['ann_id_2']} | IoU: {row['iou']:.3f}")
else:
    print("    No duplicate annotations detected.")

# -------------------- IoU Distribution Visualization --------------------
# Sample IoU values for visualization (all pairs within same image/category)
print("\n Computing IoU distribution for same-category pairs (sampled)...")

sample_ious = []
max_samples = 5000

for (image_id, cat_id), group in df_anns.groupby(['image_id', 'category_id']):
    if len(group) < 2:
        continue
    
    bboxes = group['bbox'].tolist()
    
    # Sample pairs if too many
    pairs = list(combinations(range(len(bboxes)), 2))
    if len(pairs) > 10:
        pairs = pairs[:10]
    
    for i, j in pairs:
        iou = compute_iou(bboxes[i], bboxes[j])
        sample_ious.append({'iou': iou, 'category': CAT_ID_TO_NAME.get(cat_id, 'unknown')})
        
        if len(sample_ious) >= max_samples:
            break
    
    if len(sample_ious) >= max_samples:
        break

if len(sample_ious) > 0:
    df_ious = pd.DataFrame(sample_ious)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(df_ious['iou'], bins=50, color='#3498DB', edgecolor='black', alpha=0.7)
    ax.axvline(x=IOU_DUPLICATE_THRESH, color='red', linestyle='--', linewidth=2, 
               label=f'Duplicate threshold ({IOU_DUPLICATE_THRESH})')
    ax.set_xlabel('IoU', fontsize=12)
    ax.set_ylabel('Frequency', fontsize=12)
    ax.set_title('IoU Distribution (Same-Category Pairs, Same Image)', fontsize=13, fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    print(f"\n   Sampled {len(df_ious)} IoU values.")
    print(f"   Mean IoU: {df_ious['iou'].mean():.3f}")
    print(f"   IoU > {IOU_DUPLICATE_THRESH}: {(df_ious['iou'] > IOU_DUPLICATE_THRESH).sum()} samples")

<a id='negatives'></a>
## 9. Negative Sample Strategy (Core Requirement 2)

In [ ]:
# ============================================================
# CELL 8: Negative Sample Strategy
# ============================================================

print("="*60)
print(" NEGATIVE SAMPLE STRATEGY")
print("="*60)

# Identify images with annotations
images_with_annotations = set(df_anns['image_id'].unique())
all_image_ids = set(df_images['image_id'].unique())

# Empty images (no annotations)
empty_image_ids = all_image_ids - images_with_annotations
df_empty = df_images[df_images['image_id'].isin(empty_image_ids)].copy()

# Calculate ratio
total_images = len(df_images)
n_empty = len(df_empty)
n_annotated = len(images_with_annotations)
empty_ratio = n_empty / total_images

print(f"\n Current Dataset Composition:")
print(f"   Total images: {total_images:,}")
print(f"   Annotated images: {n_annotated:,} ({n_annotated/total_images*100:.1f}%)")
print(f"   Empty images (negatives): {n_empty:,} ({empty_ratio*100:.1f}%)")

# -------------------- Visualization --------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart
ax1 = axes[0]
sizes = [n_annotated, n_empty]
labels = [f'Annotated\n({n_annotated:,})', f'Empty/Negative\n({n_empty:,})']
colors = ['#3498DB', '#E74C3C']
explode = (0, 0.05)
ax1.pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%',
        shadow=True, startangle=90)
ax1.set_title('Dataset Composition', fontsize=13, fontweight='bold')

# Bar chart with target range
ax2 = axes[1]
categories = ['Current', 'Target Min', 'Target Max']
values = [empty_ratio * 100, NEGATIVE_RATIO_MIN * 100, NEGATIVE_RATIO_MAX * 100]
colors = ['#3498DB', '#2ECC71', '#2ECC71']

bars = ax2.bar(categories, values, color=colors, edgecolor='black', linewidth=1)
ax2.axhline(y=NEGATIVE_RATIO_MIN * 100, color='green', linestyle='--', linewidth=2, alpha=0.7)
ax2.axhline(y=NEGATIVE_RATIO_MAX * 100, color='green', linestyle='--', linewidth=2, alpha=0.7)
ax2.fill_between([-0.5, 2.5], NEGATIVE_RATIO_MIN * 100, NEGATIVE_RATIO_MAX * 100, 
                  color='green', alpha=0.1)
ax2.set_ylabel('Percentage (%)', fontsize=12)
ax2.set_title('Negative Sample Ratio', fontsize=13, fontweight='bold')
ax2.set_ylim(0, max(20, empty_ratio * 100 + 5))

for bar, val in zip(bars, values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
             f'{val:.1f}%', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()

# -------------------- Recommendation --------------------
print(f"\n RECOMMENDATION:")

if empty_ratio < NEGATIVE_RATIO_MIN:
    needed = int(NEGATIVE_RATIO_MIN * n_annotated / (1 - NEGATIVE_RATIO_MIN)) - n_empty
    print(f"    Current negative ratio ({empty_ratio*100:.1f}%) is BELOW target range ({NEGATIVE_RATIO_MIN*100}%-{NEGATIVE_RATIO_MAX*100}%).")
    print(f"     ADD approximately {needed:,} more empty images.")
    print(f"     Prioritize:")
    print(f"      - Pages with Bibliographies (dense, structured text)")
    print(f"      - Pages with Double-Column Text (visual similarity to tables)")
    print(f"      - Document pages with lists or structured content (confusable with tables)")
elif empty_ratio > NEGATIVE_RATIO_MAX:
    excess = n_empty - int(NEGATIVE_RATIO_MAX * n_annotated / (1 - NEGATIVE_RATIO_MAX))
    print(f"    Current negative ratio ({empty_ratio*100:.1f}%) is ABOVE target range ({NEGATIVE_RATIO_MIN*100}%-{NEGATIVE_RATIO_MAX*100}%).")
    print(f"     REMOVE approximately {excess:,} empty images, OR add more annotated samples.")
else:
    print(f"    Current negative ratio ({empty_ratio*100:.1f}%) is within target range ({NEGATIVE_RATIO_MIN*100}%-{NEGATIVE_RATIO_MAX*100}%).")
    print(f"     Ensure empty images include hard negatives (bibliographies, multi-column layouts).")

# -------------------- Display Sample Empty Images --------------------
if n_empty > 0:
    print(f"\n  Sample Empty Images ({min(9, n_empty)} shown):")
    print("   (Visually verify these include hard negatives like bibliographies/multi-column text)")
    
    sample_empty = df_empty.sample(min(9, n_empty), random_state=42)
    
    n_samples = len(sample_empty)
    n_cols = min(3, n_samples)
    n_rows = (n_samples + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4*n_cols, 4*n_rows))
    if n_samples == 1:
        axes = np.array([[axes]])
    elif n_rows == 1:
        axes = axes.reshape(1, -1)
    
    for idx, (_, row) in enumerate(sample_empty.iterrows()):
        r, c = idx // n_cols, idx % n_cols
        ax = axes[r, c]
        
        img_path = os.path.join(IMAGES_DIR, row['file_name'])
        if os.path.exists(img_path):
            img = Image.open(img_path)
            ax.imshow(img, cmap='gray')
            ax.set_title(row['file_name'], fontsize=9)
        else:
            ax.text(0.5, 0.5, f"Image not found:\n{row['file_name']}", 
                   ha='center', va='center', transform=ax.transAxes)
        ax.axis('off')
    
    # Hide empty subplots
    for idx in range(n_samples, n_rows * n_cols):
        r, c = idx // n_cols, idx % n_cols
        axes[r, c].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("\n   ⚠️  No empty images found in the dataset!")

<a id='outlier-viz'></a>
## 10. Outlier Sample Visualization

In [ ]:
# ============================================================
# CELL 9: Outlier Sample Visualization
# ============================================================

print("="*60)
print(" OUTLIER SAMPLE VISUALIZATION")
print("="*60)

def visualize_annotations(df_subset, title, max_samples=4):
    """Visualize sample images with their annotations."""
    if len(df_subset) == 0:
        print(f"\n   No samples to visualize for: {title}")
        return
    
    # Get unique images
    unique_images = df_subset['image_id'].unique()[:max_samples]
    n_samples = len(unique_images)
    
    if n_samples == 0:
        return
    
    n_cols = min(2, n_samples)
    n_rows = (n_samples + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 6*n_rows))
    if n_samples == 1:
        axes = np.array([[axes]])
    elif n_rows == 1:
        axes = axes.reshape(1, -1)
    
    for idx, img_id in enumerate(unique_images):
        r, c = idx // n_cols, idx % n_cols
        ax = axes[r, c]
        
        img_row = df_images[df_images['image_id'] == img_id].iloc[0]
        img_path = os.path.join(IMAGES_DIR, img_row['file_name'])
        
        if os.path.exists(img_path):
            img = Image.open(img_path)
            ax.imshow(img, cmap='gray')
            
            # Draw all annotations for this image that are in the subset
            img_anns = df_subset[df_subset['image_id'] == img_id]
            
            for _, ann in img_anns.iterrows():
                x, y, w, h = ann['bbox']
                cat_id = ann['category_id']
                color = CATEGORY_COLORS.get(cat_id, 'red')
                
                rect = Rectangle((x, y), w, h, linewidth=2, 
                                 edgecolor=color, facecolor='none')
                ax.add_patch(rect)
                
                # Add label
                ax.text(x, y-5, f"{ann['category_name'][:10]}", 
                       color=color, fontsize=8, fontweight='bold',
                       bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))
            
            ax.set_title(f"{img_row['file_name']}\n({len(img_anns)} flagged annotations)", fontsize=10)
        else:
            ax.text(0.5, 0.5, f"Image not found:\n{img_row['file_name']}", 
                   ha='center', va='center', transform=ax.transAxes)
        ax.axis('off')
    
    # Hide empty subplots
    for idx in range(n_samples, n_rows * n_cols):
        r, c = idx // n_cols, idx % n_cols
        axes[r, c].axis('off')
    
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

# Create legend
print("\n Category Color Legend:")
for cat_id, cat_name in CAT_ID_TO_NAME.items():
    color = CATEGORY_COLORS.get(cat_id, '#333')
    print(f"   {cat_name}: {color}")

# -------------------- Visualize Each Flagged Category --------------------

# 1. Tiny table boxes
print(f"\n" + "="*50)
print(f" TINY TABLE BOXES (<{TINY_AREA_PCT*100}% of page)")
print("="*50)
visualize_annotations(df_tiny_tables, f"Tiny Table Boxes (<{TINY_AREA_PCT*100}% of page)")

# 2. Full-page boxes
print(f"\n" + "="*50)
print(f" FULL-PAGE BOXES (>{FULL_PAGE_PCT*100}% of page)")
print("="*50)
visualize_annotations(df_fullpage, f"Full-Page Boxes (>{FULL_PAGE_PCT*100}% of page)")

# 3. Extreme aspect ratio tables
print(f"\n" + "="*50)
print(f" EXTREME ASPECT RATIO TABLES")
print("="*50)
visualize_annotations(df_extreme_ar, "Extreme Aspect Ratio Tables")

# 4. Orphaned children (if any)
if len(df_orphan_anns) > 0:
    print(f"\n" + "="*50)
    print(f" ORPHANED CHILD ANNOTATIONS")
    print("="*50)
    visualize_annotations(df_orphan_anns, "Orphaned Child Annotations (no parent table)")

# 5. Containment issues (if any)
if len(df_containment_issues) > 0:
    print(f"\n" + "="*50)
    print(f" CONTAINMENT ISSUES (children outside parent)")
    print("="*50)
    # Get the annotation rows for visualization
    containment_ann_ids = df_containment_issues['ann_id'].tolist()
    df_containment_viz = df_anns[df_anns['id'].isin(containment_ann_ids)]
    visualize_annotations(df_containment_viz, "Children Outside Parent Table")

<a id='summary'></a>
## 11. Summary Report & Recommendations

In [ ]:
# ============================================================
# CELL 10: Summary Report & Recommendations
# ============================================================

print("="*70)
print(" PHASE 1 EDA SUMMARY REPORT")
print("="*70)

# -------------------- Compile All Flagged Annotations --------------------
flagged_records = []

# Tiny tables
for _, row in df_tiny_tables.iterrows():
    flagged_records.append({
        'ann_id': row['id'],
        'image_id': row['image_id'],
        'file_name': row['file_name'],
        'category': row['category_name'],
        'issue_type': 'tiny_box',
        'details': f"area_pct={row['box_area_pct']*100:.2f}%"
    })

# Full-page boxes
for _, row in df_fullpage.iterrows():
    flagged_records.append({
        'ann_id': row['id'],
        'image_id': row['image_id'],
        'file_name': row['file_name'],
        'category': row['category_name'],
        'issue_type': 'fullpage_box',
        'details': f"area_pct={row['box_area_pct']*100:.2f}%"
    })

# Extreme AR tables
for _, row in df_extreme_ar.iterrows():
    flagged_records.append({
        'ann_id': row['id'],
        'image_id': row['image_id'],
        'file_name': row['file_name'],
        'category': row['category_name'],
        'issue_type': 'extreme_aspect_ratio',
        'details': f"AR={row['aspect_ratio']:.3f}"
    })

# Orphaned children
if len(df_orphan_anns) > 0:
    for _, row in df_orphan_anns.iterrows():
        flagged_records.append({
            'ann_id': row['id'],
            'image_id': row['image_id'],
            'file_name': row['file_name'],
            'category': row['category_name'],
            'issue_type': 'orphaned_child',
            'details': 'no parent table'
        })

# Containment issues
for _, row in df_containment_issues.iterrows():
    flagged_records.append({
        'ann_id': row['ann_id'],
        'image_id': row['image_id'],
        'file_name': row['file_name'],
        'category': row['category'],
        'issue_type': 'outside_parent',
        'details': 'child outside parent bbox'
    })

# Duplicates (flag one of each pair)
for _, row in df_duplicates.iterrows():
    flagged_records.append({
        'ann_id': row['ann_id_2'],  # Flag the second one for removal
        'image_id': row['image_id'],
        'file_name': row['file_name'],
        'category': row['category'],
        'issue_type': 'duplicate',
        'details': f"IoU={row['iou']:.3f} with ann_id={row['ann_id_1']}"
    })

df_flagged = pd.DataFrame(flagged_records)

# -------------------- Summary Table --------------------
print("\n" + "-"*70)
print(" ISSUE SUMMARY")
print("-"*70)

summary_data = [
    ['Tiny table boxes (<3%)', len(df_tiny_tables), 'Review/remove - likely mislabeled cells'],
    ['Full-page boxes (>95%)', len(df_fullpage), 'Review/remove - likely margin annotations'],
    ['Extreme AR tables', len(df_extreme_ar), 'Adjust input resolution or review'],
    ['Orphaned child annotations', len(df_orphan_anns) if len(df_orphan_anns) > 0 else 0, 'Fix hierarchy - add parent table'],
    ['Children outside parent', len(df_containment_issues), 'Review spatial alignment'],
    ['Duplicate annotations', len(df_duplicates), 'De-duplicate'],
    ['Tables without children', len(tables_without_children), 'Complete annotations if needed'],
]

df_summary = pd.DataFrame(summary_data, columns=['Issue Type', 'Count', 'Recommended Action'])
display(df_summary)

# -------------------- Negative Sample Status --------------------
print("\n" + "-"*70)
print(" NEGATIVE SAMPLE STATUS")
print("-"*70)

print(f"   Current ratio: {empty_ratio*100:.1f}%")
print(f"   Target range: {NEGATIVE_RATIO_MIN*100}% - {NEGATIVE_RATIO_MAX*100}%")

if empty_ratio < NEGATIVE_RATIO_MIN:
    needed = int(NEGATIVE_RATIO_MIN * n_annotated / (1 - NEGATIVE_RATIO_MIN)) - n_empty
    print(f"   Status: BELOW TARGET - Add ~{needed:,} empty images")
elif empty_ratio > NEGATIVE_RATIO_MAX:
    excess = n_empty - int(NEGATIVE_RATIO_MAX * n_annotated / (1 - NEGATIVE_RATIO_MAX))
    print(f"   Status: ABOVE TARGET - Remove ~{excess:,} empty images")
else:
    print(f"   Status: WITHIN TARGET")

# -------------------- Input Resolution Recommendation --------------------
print("\n" + "-"*70)
print(" YOLO INPUT RESOLUTION RECOMMENDATION")
print("-"*70)

print(f"   Image mean AR: {mean_ar:.3f}")
print(f"   Table median AR: {table_ar_median:.3f}")

# Combined recommendation
if 0.6 <= table_ar_median <= 1.5:
    final_rec = "640x640 (balanced) or 1280x1280 (high-res)"
elif table_ar_median > 1.5:
    final_rec = "1280x640 or 960x640 (landscape, for wide tables)"
else:
    final_rec = "640x960 or 640x1280 (portrait, for tall tables)"

print(f"   Recommended: {final_rec}")

# -------------------- Export Flagged Annotations --------------------
output_csv_path = os.path.join(OUTPUT_DIR, 'flagged_annotations.csv')

if len(df_flagged) > 0:
    df_flagged.to_csv(output_csv_path, index=False)
    print(f"\n Exported {len(df_flagged)} flagged annotations to:")
    print(f"   {output_csv_path}")
    
    # Show issue type breakdown
    print("\n   Breakdown by issue type:")
    for issue_type, count in df_flagged['issue_type'].value_counts().items():
        print(f"      {issue_type}: {count}")
else:
    print(f"\n No annotations flagged for review!")

# -------------------- Final Checklist --------------------
print("\n" + "="*70)
print(" PHASE 1 COMPLETION CHECKLIST")
print("="*70)

checklist = [
    ("Area distribution analyzed", True),
    ("Tiny boxes flagged (tables only)", len(df_tiny_tables) >= 0),
    ("Full-page boxes flagged", len(df_fullpage) >= 0),
    ("Aspect ratio analyzed", True),
    ("Extreme AR tables flagged", len(df_extreme_ar) >= 0),
    ("Spatial heatmap generated", True),
    ("Hierarchical consistency checked", True),
    ("IoU duplicates checked", True),
    ("Negative sample ratio evaluated", True),
    ("Outlier samples visualized", True),
    ("Flagged annotations exported", len(df_flagged) >= 0),
]

for task, done in checklist:
    status = "OK" if done else "NOK"
    print(f"   {status} {task}")

print("\n" + "="*70)
print(" READY FOR PHASE 2: DATA CONVERSION")
print("="*70)
print("\n   Next steps:")
print("   1. Review flagged annotations in 'flagged_annotations.csv'")
print("   2. Clean dataset by removing/correcting problematic annotations")
print("   3. Add hard-negative empty images if needed")
print("   4. Proceed to Phase 2: COCO -> YOLO format conversion")